# Cloud phase over the whole region — all cells

Companion to `plot_lwp_histogram_by_surface_class_fraction_limits.ipynb`, which
breaks the same analysis up by surface class. This notebook keeps the views that
pool **every cell in the region** into one number.

Both notebooks call the same module with the same options, so a figure here and
the matching per-class panel there come from one code path — only the set of
cells averaged over differs.

One caveat when reading a whole-domain number: the Barrow strip is not
homogeneous, and its composition changes through the season — roughly 87% open
ocean in September against 87% sea ice by March. A domain average is therefore
an average over a *changing mixture* of surfaces, not over a fixed one. Where
that matters, the per-class notebook is the one to read.

## Setup

In [25]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt

import plot_lwp_histogram_by_surface_class as lwph

# open_mfdataset warns about join/compat defaults changing; the module pins
# both explicitly, so the warning is noise here.
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.dpi"] = 110      # on-screen only; saved files use --dpi

# Set to a directory to write PNGs as well as display them; None displays only.
# SAVE_DIR = None
SAVE_DIR = Path("figures/fraction_limits_all_cells")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load the data

The slow cell. Same options as the per-class notebook.

In [ ]:
A = lwph.prepare(
    region="barrow",
    years=tuple(range(2022, 2026)),   # season START years
    season_start=(10, 1),
    season_end=(3, 31),

    phase_mode="fraction",            # <- the fractional scheme

    liquid_fraction_min=0.90,         # LWP/CWP at or above this = liquid only
    ice_fraction_min=0.90,            # IWP/CWP at or above this = ice only
                                      # (must sum to more than 1.0)

    min_lwp=0.1,                      # g m-2, liquid below this counts as absent
    min_iwp=0.1,                      # g m-2, ice below this counts as absent

    min_cloud_fraction=0.99,                         # overcast only
    lsm_tol = 0.01,                                 # land-sea mask tolerance for 
                                                    # classifying a grid cell as land or sea
    open_ocean_max_siconc = 0.05,                       # maximum sea ice concentration for a grid cell to be
                                                    # considered open ocean
    sea_ice_min_siconc = 0.95,                          # minimum sea ice concentration for a grid cell to be
                                                    # considered sea ice,     
    storage="local",
    dpi=350,
)

Liquid-bearing cloud hours by LWP and surface class
  Source     : /Users/andrewbuggee/Documents/VS_CODE/Python-Research/ERA5/surface_energy_budget/data/barrow
  Grid       : 41 x 61 cells, 107,856 time steps
  Season     : 08-01 to 03-31  (wraps the new year)
  Classes    : lsm tol 0.01 | open ocean < 0.05 | pack ice > 0.95
  Cloudy     : tcc >= 0.99
  Phase mode : fraction
  Categories : liquid only: LWP/CWP >= 0.9 | ice only: IWP/CWP >= 0.9 | mixed: the rest | CWP = LWP + IWP above 0.1/0.1 g m-2
               ice only: IWP/CWP >= 0.9
  LWP bins   : linear 0-600 in 24 | log 0.1-1000 in 12

  Seasons found (18), coverage of the 244-day window:
    1999/2000:  37.3%   (below --min-season-coverage)
    2000/2001:  99.6%
    2001/2002:  62.7%
    2011/2012: 100.0%
    2012/2013:  99.6%
    2013/2014:  99.6%
    2014/2015:  99.6%
    2015/2016: 100.0%
    2016/2017:  99.6%
    2017/2018:  95.1%
    2018/2019:  99.6%
    2019/2020: 100.0%
    2020/2021:  99.6%
    2021/2022:  99.6%
    20

## Numeric report

In [ ]:
lwph.print_report(A)

## Monthly cloud-phase occupancy, all cells

One panel per month; three bars giving the share of that month's cell-hours
spent under a liquid-only, mixed-phase, or ice-only cloud, averaged over
seasons.

The three bars do **not** sum to 100%: they are three of the four categories
partitioning the overcast hours (the fourth is the report's "neither"), and
clear hours are in none of them. Each panel's subtitle states the overcast
share, so the gap is accounted for rather than left to the reader.

In [ ]:
lwph.fig_monthly_phase_fraction(A, out_dir=SAVE_DIR);

## The ARM cell, for comparison

`fig_monthly_phase_fraction` takes `surface_class=` — `None`/`"all"` for the
whole domain, any name in `CLASS_ORDER`, or `"arm_site"`. Kept here so the
single-cell comparison sits beside the domain average without switching
notebooks.

In [ ]:
lwph.fig_monthly_phase_fraction(A, surface_class="arm_site", out_dir=SAVE_DIR);